 
<img src="https://th.bing.com/th/id/R.3cd1c8dc996c5616cf6e65e20b6bf586?rik=09aaLyk4hfbBiQ&riu=http%3a%2f%2fcidics.uanl.mx%2fwp-content%2fuploads%2f2016%2f09%2fcimat.png&ehk=%2b0brgMUkA2BND22ixwLZheQrrOoYLO3o5cMRqsBOrlY%3d&risl=&pid=ImgRaw&r=0" 
     style="float: right; margin-right: 30px;" 
     width="120"
     />

---
 
# **PROCESAMIENTO DEL LENGUAJE NATURAL: TAREA 5: Modelos de Lenguaje Neuronales.**
EZAU FARIDH TORRES TORRES.
===
     
<p align="right"> Maestría en Ciencias con Orientación en Matemáticas Aplicadas. </p>
<p align="right"> CENTRO DE INVESTIGACIÓN EN MATEMÁTICAS. </p>
<p align="right"> Fecha de entrega: 18/03/2025. </p>


---

In [13]:
# Necessary libraries.
import numpy as np                             # library for numerical operations.
import nltk                                    # library for text processing.
from nltk.tokenize import TweetTokenizer       # library for tweet tokenization.
from nltk.corpus import stopwords              # library for stopwords.
stopwords_es = set(stopwords.words('spanish')) # set of spanish stopwords.
import os                                      # library for operating system.
import time                                    # library for time.
import shutil                                  # library for file operations.
from typing import Tuple, Callable             # library for typing.
import random                                  # library for random numbers.
import pandas as pd                            # library for data manipulation.
from typing import Tuple                       # library for typing.
import torch                                   # library for tensor operations.
from tqdm import tqdm                          # library for progress bar.
import torch.nn as nn                          # library for neural network operations.
from torch.utils.data import (DataLoader,      # library for data loading.
                              TensorDataset)   # library for tensor dataset.
import torch.nn.functional as F                # library for neural network operations.
from sklearn.metrics import accuracy_score     # library for accuracy score.
from itertools import permutations             # library for permutations.
from colorama import Fore, Style               # library for colored text.
np.random.seed(0)                              # seed for reproducibility.
random.seed(0)                                 # seed for reproducibility.
torch.manual_seed(0)                           # seed for reproducibility.
torch.backends.cudnn.benchmark = False         # reproducibility.

> Primero se cargan los datos de entrenamiento y de validación en forma de lista (con cada tweet como un string). Cada elemento es un tweet, exactamente como en la práctica del Modelo Neuronal de Bengio.

In [2]:
X_train = pd.read_csv('./data/mex20_train.txt', sep = '\r\n', header = None, engine = 'python').loc[:,0].values.tolist()
X_val   = pd.read_csv('./data/mex20_val.txt', sep = '\r\n', header = None, engine = 'python').loc[:,0].values.tolist()
print("First 5 elements in Train data:\n", X_train[:5])
print("First 5 elements in Validation data:\n", X_val[:5])

First 5 elements in Train data:
 ['@USUARIO @USUARIO @USUARIO Q se puede esperar del maricon de closet de la Yañez aun recuerdo esa ves q lo vi en zona rosa viendo quien lo levantada', '@USUARIO La piel nueva siempre arde un poquito los primeros días... y más con este puto clima', 'Ustedes no se enamoran de mí… por tontas.', 'Me las va a pagar esa puta gorda roba tuits...', '@USUARIO LA GENTE ES TONTA PORQUE NO SE DAN CUENTA QUE TÚ HACES A BATMAN AZUL']
First 5 elements in Validation data:
 ['Al perro que se te acerque le parto su madre a si de facil', '@USUARIO @USUARIO Él supo sacar a su familia adelante en lo que sabe en el mundo existen muchas personas ardidas como tú 🤷🏻\u200d♀️', '@USUARIO Entonces para que quieres estar en sus paises?, mejor vente aca y chinguele cabrona de verdad, maldita sangana.', 'Que bueno que hoy juega México, porque tú vales verga, hija de la chingada.', 'Ojalá un día me valgas la misma verga que al vato de la cfe despelucando y ponchando a medio del puent

* * *
# **Instrucciones**

Realiza los siguientes puntos en un notebook de Python *lo mejor organizado y claro posible*. Ponga su nombre completo al archivo de entrega (e.g., adrian_pastor_lopez_monroy.ipynb) y también en la primera celda del notebook junto con el número de Tarea. Al entregar la tarea, sube al classroom el notebook como un archivo (NO zip, ni rar, etc.). El notebook deberá haber sido ejecutado en tú máquina (o colab) y mostrar el resultado en las celdas.

Se vale pedir ayuda y/o copiar con atribución entre los miembros de la clase y apegándose estrictamente a los siguientes puntos:
1. Del total de actividades que se solicitan hacer (2 con valor para esta tarea) solo puedes pedir ayuda/copiar en un total de **una**.
2. Para los puntos dónde se pide ayuda, brevemente escribe en qué pediste ayuda y a quién.
3. Si tuviste que reusar alguna parte de código que no es tuyo, deja claro dos cosas: 1) brevemente porque tuviste dificultad para hacerlo, 2) cómo lo solucionó tu compañero.

* * *
# **1.- Modelos de Lenguaje Neuronales**

## **1.1- (50pts) Con base en la implementación mostrada en la práctica del modelo de Bengio,**
1. **Construya un modelo de lenguaje neuronal a nivel de palabra, pero preinicializado con los embeddings proporcionados. Tomé en cuenta secuencias de tamaño 4 para el modelo, es decir hasta 3 palabras en el contexto.**
2. **Después de haber entrenado el modelo, recupere las 10 palabras más similares a tres palabras de su gusto dadas.**
3. **Ponga al modelo a generar texto a partir de tres secuencias de inicio de su gusto.**
4. **Escriba 5 ejemplos de oraciones y mídales el likelihood.**
5. **Proponga un ejemplo para ver estructuras sintácticas (permutaciones de palabras de alguna oración) buenas usando el likelihood a partir de una oración que usted proponga.**
6. **Calcule la perplejidad del modelo sobre los datos val. Compárelo con la perplejidad del modelo de lenguaje sin embeddings preentrenados y el probabilista de la tarea anterior (el visto en clase). DISCUTA BREVEMENTE.**

> Se carga el archivo de embeddings *word2vec_col.txt* y se crea el diccionario *pretrained_embeddings_dict* donde las keys es la palabra:

In [3]:
def load_embeddings_file(file: str) -> dict:
    """
    Load the embeddings file. The file should be a text file where each line contains a word and its vector.
    A dictionary is returned where the key is the word and the value is the vector.
    
    Parameters
    ----------
    file : str
        The file path.

    Returns
    -------
    dict
        Pretrained embeddings. The key is the word and the value is the vector.
    """
    pretrained_embeddings_dict = {}                   # Dictionary to store the embeddings.
    with open(file, 'r', encoding = 'utf-8') as f:    # Open the file.
        for line in f:                                # Read the file line by line.
            sections = line.strip().split()           # Separate the word and the vector.
            word = sections[0]                        # Get the word.
            vector = list(map(float, sections[1:]))   # Get the vector.
            pretrained_embeddings_dict[word] = vector # Store the word and the vector.
    return pretrained_embeddings_dict                 # Return the pretrained embeddings.

# Load the embeddings file.
pretrained_embeddings_dict = load_embeddings_file('./data/word2vec_col.txt')
print('Cantidad de palabras en los embeddings:', len(pretrained_embeddings_dict))
print('Dimensiones de los embeddings         :', len(list(pretrained_embeddings_dict.values())[0]))
print('Primeras 7 palabras en los embeddings :', list(pretrained_embeddings_dict.keys())[:10], '...')

Cantidad de palabras en los embeddings: 973265
Dimensiones de los embeddings         : 100
Primeras 7 palabras en los embeddings : ['de', 'que', 'la', 'a', 'y', 'el', 'no', 'en', 'me', 'to'] ...


> A continuación, se construye un modelo de lenguaje neuronal a nivel de palabra con la opción de preinicializar con los embeddings preentrenados.
>
>Primero, se define la clase *WordNgramData* en la cual se preparan los datos para el modelo de ngrama. Es muy similar a la de la práctica y destaca la opción de agregar un diccionario de embeddings pre-entrenamos. Si no se le da un modelo de embedddings, la matriz de embeddings se aprende en el proceso de entrenamiento. Además, tiene la opción de eliminar las stopwords como variable booleana. Cuenta con los métodos:
>- *default_tokenizer()*
>- *remove_word()*
>- *get_vocab()*
>- *fit()*
>- *get_ngram_doc()*
>- *transform()*
>
> (Se describe cada uno en su docstring).
>
> **NOTA:** Nos surge la duda: ¿Cómo manejar las palabras que están en el corpus de entrenamiento pero no en el conjunto de embeddings preentrenados (w2v)? Podríamos directamente mapearlas al token $<unk>$, pero podríamos perder mucha información si se tienen demasiadas palabras en esta situación. Yo preferí entrenar desde cero las palabras que están en el vocabulario del corpus pero no en los embeddings para que el modelo aprenda representaciones significativas para estas palabras durante el entrenamiento. Además, agregué una inicialización explícita del token $<unk>$ para que este no se entrene de forma completamente aleatoria y tenga una representación estable (usé el vector promedio para que su representación se mantenga en una región del espacio de embeddings coherente). Esto se hace en el método *fit()* de la clase *WordNgramData*:


In [4]:
class WordNgramData:
    """
    Word-level N-gram data preparation class. It prepares the data for the n-gram model. It creates the vocabulary,
    the word to index and index to word mappings, and the embeddings matrix. It also transforms the corpus
    into n-grams.
    """
    def __init__(self, model_order: int = 4, max_vocab_size: int = 5000, tokenizer: Callable = None,
                 embeddings_model: dict = None, remove_stopwords: bool = False):
        """
        Constructor method. If no tokenizer is provided, the default tokenizer is used. The default tokenizer
        splits the document by spaces. If no embeddings model is provided, the embeddings matrix is trained during
        the training process. If remove_stopwords is True, stopwords are removed.
        
        Parameters
        ----------
        model_order : int, optional
            Order of the n-gram model. Default is 4.
        max_vocab_size : int, optional
            Maximum vocabulary size (default is 5000).
        tokenizer : Callable, optional
            Tokenizer method. Default is None.
        embeddings_model : dict, optional
            Embeddings model. Default is None. An embeddings model is a dictionary where the keys are words and the values are the embeddings.
        remove_stopwords : bool, optional
            Remove stopwords. Default is False.
        """
        self.tokenizer = tokenizer or self.default_tokenizer  # tokenizer object.
        self.punct = {'.',',',';',':','!','?','¿','¡','(',')','[',']','{','}',          # punctuation.
                      '"',"'",'...','…','“','”','‘','’','—','-','-','<url>','@usuario'}
        self.model_order = model_order                        # n-gram model order.
        self.max_vocab_size = max_vocab_size                  # maximum vocabulary size.
        self.UNK, self.SOS, self.EOS = "<unk>", "<s>", "</s>" # special tokens.
        self.embeddings_model = embeddings_model              # embeddings model.
        self.remove_stopwords = remove_stopwords              # remove stopwords.
        self.vocab, self.w2id, self.id2w = None, None, None   # vocabulary, word to index, index to word mappings.
        self.embeddings_matrix = None                         # embeddings matrix.

    @staticmethod
    def default_tokenizer(doc: str) -> list:
        """
        Default tokenizer method. It splits the document by spaces.
        """
        return doc.split(" ")

    def remove_word(self, word: str) -> bool:
        """
        Remove word method. It removes punctuation and numbers. If remove_stopwords is True, it also removes stopwords.
        
        Parameters
        ----------
        word : str
            Word to remove.
        
        Returns
        -------
        bool
            True if the word is to remove, False otherwise.
        """
        word_lower = word.lower() # lowercase word.
        if self.remove_stopwords: # remove stopwords.
            return word_lower in self.punct or word_lower.isnumeric() or word_lower in stopwords_es
        else:
            return word_lower in self.punct or word_lower.isnumeric()

    def get_vocab(self, corpus: list) -> set:
        """
        Get vocabulary method in lowercase. It gets the vocabulary from the corpus based on the frequency distribution
        and the maximum vocabulary size. It also removes punctuation, numbers, and stopwords.

        Parameters
        ----------
        corpus : list
            List of documents.

        Returns
        -------
        set
            Set of words.
        """
        freq_dist = nltk.FreqDist(       # frequency distribution.
            w.lower() for doc in corpus  # lowercase document.
            for w in self.tokenizer(doc) # tokenize document.
            if not self.remove_word(w)   # remove word.
        )
        sorted_words = sorted(freq_dist, key = freq_dist.get, reverse = True)[:self.max_vocab_size - 3] # sorted words. -3 for special tokens.
        return set(sorted_words)

    def fit(self, corpus: list) -> None:
        """
        Fit method. Here we get the vocabulary and create the word to index and index to word mappings.
        If an embeddings model is provided, we also create the embeddings matrix. If a word is not in the embeddings model,
        we create a random embedding for it and it is trained during the training process.

        Parameters
        ----------
        corpus : list
            List of documents.
        """
        self.vocab = self.get_vocab(corpus) | {self.UNK, self.SOS, self.EOS} # vocabulary and add special tokens.
        self.w2id = {word: idx for idx, word in enumerate(self.vocab)}       # word to index.
        self.id2w = {idx: word for word, idx in self.w2id.items()}           # index to word.

        # If embeddings model is provided, create embeddings matrix.
        if self.embeddings_model is not None:
            embedding_size = len(next(iter(self.embeddings_model.values())))               # get embedding size.
            unk_vector = np.mean(np.array(list(self.embeddings_model.values())), axis = 0) # unknown word vector.

            self.embeddings_matrix = np.array([
                self.embeddings_model[word] if word in self.embeddings_model 
                else np.random.rand(embedding_size)                          # random embeddings for unknown words.
                for word in self.vocab
            ])
            self.embeddings_matrix[self.w2id[self.UNK]] = unk_vector         # set unknown word vector.

    def get_ngram_doc(self, doc: str) -> list:
        """
        Get n-gram document method. It gets the n-grams from the document.
        <s> padding before and </s> padding after each sentence.
        
        Parameters
        ----------
        doc : str
            Document.
            
        Returns
        -------
        list
            List of n-grams.
        """
        tokens = [token if token in self.vocab else self.UNK             # unknown token.
                  for token in [w.lower() for w in self.tokenizer(doc)]] # lowercase and tokenization.
        tokens = (self.model_order - 1)*[self.SOS] + tokens + [self.EOS] # add start and end tokens.
        return list(nltk.ngrams(tokens, self.model_order))               # n-grams.

    def transform(self, corpus: list) -> Tuple[np.ndarray, np.ndarray]:
        """
        Transform method. Here we transform the corpus into n-grams.
        It builds the X (features) and y (labels) n-grams.
        
        Parameters
        ----------
        corpus : list
            List of documents.
    
        Returns
        -------
        Tuple[np.ndarray, np.ndarray]
            Tuple with X and y n-grams.
        """
        X_ngrams, y_ngrams = [], []                                     # n-grams.
        for doc in corpus:                                              # for each document.
            for words_window in self.get_ngram_doc(doc):                # for each n-gram.
                words_window_ids = [self.w2id[w] for w in words_window] # word to index.
                X_ngrams.append(words_window_ids[:-1])                  # X n-gram.
                y_ngrams.append(words_window_ids[-1])                   # y n-gram.    
        return np.array(X_ngrams), np.array(y_ngrams)                   # return X and y n-grams.

### **1.1.1.**

>Aquí, se implementa la clase *NeuralLanguageModel* de *nn.Module*, la cual describe la estructura principal de la red, se eligió la misma que en la práctica y el paper de Bengio, i.e., solo una capa oculta.
>
> Con ayuda de la capa de *Embeddings*, primero se verifica si hay embeddings preentrenados disponibles, si sí se tienen, se toma en cuenta su tamaño y se crea una capa de embeddings usando *nn.Embedding.from_pretrained()*, que permite cargar directamente los embeddings preentrenados en formato tensorial. Aquí, *freeze=False* indica que los embeddings se entrenarán junto con el resto del modelo (este elegí yo). Si *freeze=True*, los embeddings permanecerían fijos durante el entrenamiento. Si no se tienen embeddings preentrenados, se inicializan aleatoriamente y se entrenan desde cero.

In [6]:
class NeuralLanguageModel(nn.Module):
    """
    Neural language model class.
    """
    def __init__(self, params: dict, pretrained_embeddings: np.ndarray = None):
        """
        Constructor method to initialize the neural language model.

        Parameters
        ----------
        params : dict
            Dictionary with parameters.
        pretrained_embeddings : np.ndarray, optional
            Matrix with pretrained embeddings. Default is None. If None, embeddings are trained during the
            training process and the matrix is initialized randomly, otherwise, the matrix is initialized
            with the pretrained embeddings.

        Dictionary params
        -----------------
        vocab_size : int
            Vocabulary size.
        model_order : int
            Order of the n-gram model.
        embedding_size : int
            Vector size for the embeddings (each word is represented by a dense vector).
        hidden_size : int
            Number of neurons in the hidden layer.
        dropout : float
            Dropout.
        """
        super().__init__()
        self.window_size = params['model_order'] - 1 # window size (context).

        # Embeddings layer.
        if pretrained_embeddings is not None:                                              # If we have pretrained embeddings.
            self.embedding_size = pretrained_embeddings.shape[1]                           # Get the embedding size.
            self.embeddings = nn.Embedding.from_pretrained(                                # Initialize the embeddings.
                torch.tensor(pretrained_embeddings, dtype = torch.float32), freeze = False # Freeze the embeddings, i.e., they are trained.
            )                                                     
        else:                                                                         # If we don't have pretrained embeddings.                    
            self.embedding_size = params['embedding_size']                            # Get the embedding size.    
            self.embeddings = nn.Embedding(params['vocab_size'], self.embedding_size) # Initialize the embeddings randomly.

        # Linear layers apply transformations to predict the next word.
        self.fc1 = nn.Linear(self.window_size * self.embedding_size, params['hidden_size'])
        # Dropout layer to prevent overfitting.
        self.drop1 = nn.Dropout(p = params['dropout'])
        # Output layer.
        self.fc2 = nn.Linear(params['hidden_size'], params['vocab_size'], bias = False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward method. Embeds input tokens into dense vectors. Reshapes them before feeding
        into the dense layers. Uses ReLU activation and Dropout before the final prediction.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
            
        Returns
        -------
        torch.Tensor
            Output tensor. The model outputs raw scores (logits) for each word in the vocabulary.
        """
        x = self.embeddings(x).view(-1, self.window_size * self.embedding_size) # Embeddings.
        
        return self.fc2(self.drop1(F.relu(self.fc1(x))))                        # Forward pass.

>Se define la clase *UtilsForEvaluation*, en la cual se anexaron los métodos:
>- *get_preds()*
>- *model_eval()*
>
>usados para el entrenamiendo y la evaluación de las predicciones, con la misma utilidad que en la práctica (se describe cada uno en su docstring).

In [7]:
class UtilsForEvaluation:
    """
    Utility class for evaluation.
    """
    @staticmethod
    def get_preds(raw_logits: torch.Tensor) -> np.ndarray:
        """
        Get predictions method. Here we get the predictions from the raw logits (output of the neural network).
        Applies softmax to logits to obtain probabilities. Uses torch.argmax() to select the predicted class.

        Parameters
        ----------
        raw_logits : torch.Tensor
            Raw logits, i.e., the output of the neural network.

        Returns
        -------
        np.ndarray
            Predictions, i.e., the word with the highest probability.
        """
        probs = F.softmax(raw_logits.detach(), dim = 1)   # probabilities.
        return torch.argmax(probs, dim = 1).cpu().numpy() # predictions.

    @staticmethod
    def model_eval(data: DataLoader, model: nn.Module, gpu: bool) -> float:
        """
        Model evaluation method. Iterates over batches in the validation set.
        Accumulates predictions and computes accuracy using sklearn.

        Parameters
        ----------
        data : DataLoader
            Data loader.
        model : nn.Module
            Neural network model.
        gpu : bool
            If GPU is used.

        Returns
        -------
        float
            Accuracy score.
        """
        with torch.no_grad():                                          # no gradients.
            preds, targets = [], []                                    # predictions and targets.
            for window_words, labels in data:                          # for each batch.
                if gpu:                                                # if GPU is used.
                    window_words = window_words.cuda()                 # send to GPU.
                raw_logits = model(window_words)                       # model output.
                preds.append(UtilsForEvaluation.get_preds(raw_logits)) # predictions.
                targets.append(labels.numpy())                         # targets.
                
        return accuracy_score([e for l in preds for e in l],[e for l in targets for e in l])

>Como parte del modelo de red neuronal, finalmente se define la clase *Trainer*, la cual implementa la fase de entrenamiento de la red, se usó lo mismo que en la práctica:
>- CrossEntropyLoss como métrica,
>- El optimizador SGD,
>- ReduceLROnPlateau como learning rate scheduler
>
>Recibe como parámetros al modelo, y los loaders de entrenamiento y validación, además de un diccionario de parámetros para el entrenamiento definidos en el docstring. Incluye los métodos: *save_checkpoint()* y *train()*.

In [8]:
class Trainer:
    """
    Trainer class.
    """
    def __init__(self, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, params: dict,
                 checkpoint_path: str = "model", checkpoint_filename: str = "checkpoint.pth", best_model_filename: str = "model_best.pth"):
        """
        Constructor method for the Trainer class. It uses CrossEntropyLoss for multi-class classification and SGD optimizer.
        The learning rate scheduler is ReduceLROnPlateau (reduces learning rate when a metric has stopped improving).
        Uses patience-based early stopping: If no improvement occurs for several epochs, training halts.

        Parameters
        ----------
        model : nn.Module
            Neural network model.
        train_loader : DataLoader
            Training data loader.
        val_loader : DataLoader
            Validation data loader.
        params : dict
            Dictionary with parameters.
        checkpoint_path : str, optional
            Checkpoint path. Default is "model".
        checkpoint_filename : str, optional
            Filename. Default is "checkpoint.pth".
        best_model_filename : str, optional
            Filename. Default is "model_best.pth".

        Dictionary params
        -----------------
        lr : float
            Learning rate.
        num_epochs : int
            Number of epochs.
        patience : int
            Patience, i.e., early stopping threshold.
        lr_patience : int
            Learning rate patience.
        lr_factor : float
            Learning rate factor.
        savedir : str
            Save directory.
        use_gpu : bool
            Use GPU.
        """
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.params = params
        self.checkpoint_path = checkpoint_path
        self.checkpoint_filename = checkpoint_filename
        self.best_model_filename = best_model_filename
        self.criterion = nn.CrossEntropyLoss()                                  # loss function.
        self.optimizer = torch.optim.SGD(model.parameters(), lr = params['lr']) # optimizer.
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(            # scheduler.
            self.optimizer,                   # optimizer to adjust.
            mode = 'min',                     # mode (min means reduce when the quantity monitored has stopped decreasing).
            patience = params['lr_patience'], # patience (number of epochs with no improvement after which learning rate will be reduced).
            factor = params['lr_factor']      # factor by which the learning rate will be reduced. new_lr = lr * factor.
        )

        # Metrics (in this case, accuracy).
        self.train_metric_history = [] # training metric history.
        self.val_metric_history = []   # validation metric history.
        self.best_val_metric = 0       # best validation accuracy.

    def save_checkpoint(self, state: dict, is_best: bool) -> None:
        """
        Save checkpoint method, i.e., saves the model's state during training. 

        Parameters
        ----------
        state : dict
            Dictionary with state, i.e., model state_dict, optimizer state_dict, epoch, loss, and accuracy.
        is_best : bool
            If is the best model, i.e., the model with the highest accuracy.
        checkpoint_path : str
            Checkpoint path.
        filename : str, optional
            Filename. Default is "checkpoint.pth".
        """
        if not os.path.exists(self.checkpoint_path):                            # if directory does not exist.
            os.makedirs(self.checkpoint_path, exist_ok = True)                  # create directory.
        filepath = os.path.join(self.checkpoint_path, self.checkpoint_filename) # file                  
        torch.save(state, filepath)                                             # save checkpoint.                 
        if is_best:                                                             # if is the best model.   
            shutil.copyfile(filepath, os.path.join(self.checkpoint_path, self.best_model_filename)) # copy the best model.
    
    def train(self):
        """
        Train method. Here we train the neural network model.

        Description
        -----------
        The training process is as follows:
        1. We iterate over the number of epochs.
        2. For each epoch, we iterate over the training data.
        3. We calculate the loss and the accuracy.
        4. We update the weights.
        5. We evaluate the model on the validation data.
        6. We save the best model.
        7. We print the training and validation metrics.
        8. We break the training process if there is no improvement.
        9. We print the total time.
        """
        start_time = time.time() # start time.
        n_no_improve = 0         # number of epochs with no improvement.

        for epoch in tqdm(range(self.params['num_epochs']), desc = "Training..."):
            
            epoch_time = time.time() # epoch time.
            loss_epoch_batch = []   # Tracks the total loss for each batch in the current epoch.
            train_metric_batch = [] # Tracks the training accuracy for each batch in the current epoch.
            self.model.train()      # set model to training mode.
            
            for window_words, labels in self.train_loader:
                
                if self.params['use_gpu']:
                    window_words, labels = window_words.cuda(), labels.cuda()

                raw_logits = self.model(window_words)     # model output.
                loss = self.criterion(raw_logits, labels) # calculate loss.
                loss_epoch_batch.append(loss.item())      # append loss.

                preds = UtilsForEvaluation.get_preds(raw_logits)          # get predictions given the raw logits.
                targets = labels.cpu().numpy()                            # get targets.
                train_metric_batch.append(accuracy_score(targets, preds)) # append accuracy.

                self.optimizer.zero_grad() # zero gradients.
                loss.backward()            # backpropagation.
                self.optimizer.step()      # update weights.

            mean_train_metric = np.mean(train_metric_batch)     # mean training accuracy.
            self.train_metric_history.append(mean_train_metric) # append training accuracy.

            self.model.eval()                                                                               # set model to evaluation mode.
            val_metric = UtilsForEvaluation.model_eval(self.val_loader, self.model, self.params['use_gpu']) # calculate validation accuracy.
            self.val_metric_history.append(val_metric)                                                      # append validation accuracy.

            self.scheduler.step(val_metric) # adjust learning rate.

            # Check if there is an improvement.
            is_improvement = val_metric > self.best_val_metric
            if is_improvement:
                self.best_val_metric = val_metric
                n_no_improve = 0
            else:
                n_no_improve += 1

            # Save checkpoint.
            self.save_checkpoint(
                state =
                {
                    'epoch'           : epoch + 1,
                    'state_dict'      : self.model.state_dict(),
                    'best_val_metric' : self.best_val_metric,
                    'optimizer'       : self.optimizer.state_dict(),
                    'scheduler'       : self.scheduler.state_dict(),
                },
                is_best = is_improvement
            )

            # Early stopping.
            if n_no_improve >= self.params['patience']:
                tqdm.write(f"No improvement. Breaking at epoch {epoch}")
                break
            
            total_time = time.time() - start_time # total time.
            tqdm.write(f"{Style.BRIGHT} Epoch{epoch + 1:>3}/{self.params['num_epochs']}{Style.RESET_ALL}"
                       f"  ➤ Train acc: {Fore.GREEN}{mean_train_metric:.4f}{Style.RESET_ALL} | "
                       f"Loss: {Fore.YELLOW}{np.mean(loss_epoch_batch):.4f}{Style.RESET_ALL} | "
                       f"Val acc: {Fore.CYAN}{val_metric:.4f}{Style.RESET_ALL} | "
                       f"Epoch Time: {time.time() - epoch_time:.2f} s | "
                       f"Total Time: {total_time:.2f} s")
    
        print(f"\n{Style.BRIGHT}--- Total Training Time: {total_time:.2f} seconds ---{Style.RESET_ALL}")

>Ahora sí, vamos a realizar las actividades del ejercicio. 
>
>Comenzamos creando el tokenizador, se fija el tamaño máximo del vocavulario en $5000$, el orden del modelo de Ngrama en $4$ para tener hasta $3$ palabras de contexto. Para la creación de los batchs, se elige un tamaño de $64$:

In [9]:
# Parameters for the NgramData class.
tokenizer = TweetTokenizer(preserve_case = False, reduce_len = True, strip_handles = True) # Tokenizer.
max_vocab_size = 5000 # maximum vocabulary size.
model_order    = 4    # n-gram model order.
batch_size     = 64   # batch size.

> Dado esto, se crean las instancias de los modelos de Ngramas con y sin los embeddings preentrenados, estos datos pasan por los métodos *fit()* y *transform()* respectivos y se crean sus Data Loaders para datos de entrenamiento y validación:

In [10]:
# ------------------------------------------------------
# NgramData WITH pretrained embeddings.
# ------------------------------------------------------
ngram_data_emb = WordNgramData(                   # NgramData with embeddings.
    model_order      = model_order,               # model order = 4 (3 for context).
    max_vocab_size   = max_vocab_size,            # maximum vocabulary size.
    tokenizer        = tokenizer.tokenize,        # tokenizer.
    embeddings_model = pretrained_embeddings_dict # pretrained embeddings.
)

# Fit the NgramData WITH embeddings and transform the data for the model with embeddings.
ngram_data_emb.fit(X_train)                                                # fit the data.         
X_ngrams_train_emb, y_ngrams_train_emb = ngram_data_emb.transform(X_train) # transform train data.
X_ngrams_val_emb, y_ngrams_val_emb = ngram_data_emb.transform(X_val)       # transform validation data.
print("Embeddings matrix shape:", ngram_data_emb.embeddings_matrix.shape)  # embeddings matrix shape.

# Dataset and DataLoader for the model with embeddings.
train_dataset_emb = TensorDataset(torch.tensor(X_ngrams_train_emb), torch.tensor(y_ngrams_train_emb)) # train dataset.
train_loader_emb = DataLoader(dataset = train_dataset_emb, # training data loader.
                              batch_size = batch_size,     # batch size.
                              shuffle = True)              # shuffle.
                           
val_dataset_emb = TensorDataset(torch.tensor(X_ngrams_val_emb), torch.tensor(y_ngrams_val_emb)) # validation dataset.
val_loader_emb = DataLoader(dataset = val_dataset_emb,     # validation data loader.
                            batch_size = batch_size,       # batch size.
                            shuffle = False)               # no shuffle.
                          

Embeddings matrix shape: (5000, 100)


In [11]:
# ------------------------------------------------------
# NgramData WITHOUT pretrained embeddings.
# ------------------------------------------------------
ngram_data_no_emb = WordNgramData(         # NgramData without embeddings.
    model_order      = model_order,        # model order = 4 (3 for context).
    max_vocab_size   = max_vocab_size,     # maximum vocabulary size.
    tokenizer        = tokenizer.tokenize, # tokenizer.
    embeddings_model = None                # no pretrained embeddings.
)

# Fit the NgramData WITHOUT embeddings and transform the data for the model with embeddings.
ngram_data_no_emb.fit(X_train)                                                      # fit the data.         
X_ngrams_train_no_emb, y_ngrams_train_no_emb = ngram_data_no_emb.transform(X_train) # transform train data.
X_ngrams_val_no_emb, y_ngrams_val_no_emb = ngram_data_no_emb.transform(X_val)       # transform validation data.
print("Embeddings matrix shape:", ngram_data_no_emb.embeddings_matrix)              # embeddings matrix shape.

# Dataset and DataLoader for the model WITHOUT embeddings.
train_dataset_no_emb = TensorDataset(torch.tensor(X_ngrams_train_no_emb), torch.tensor(y_ngrams_train_no_emb)) # train dataset.
train_loader_no_emb = DataLoader(dataset = train_dataset_no_emb, # training data loader.
                                 batch_size = batch_size,        # batch size.
                                 shuffle = True)                 # shuffle.
val_dataset_no_emb = TensorDataset(torch.tensor(X_ngrams_val_no_emb), torch.tensor(y_ngrams_val_no_emb)) # validation dataset.
val_loader_no_emb = DataLoader(dataset = val_dataset_no_emb, # validation data loader.
                               batch_size = batch_size,      # batch size.
                               shuffle = False)              # no shuffle.

Embeddings matrix shape: None


>Se usan los mismos hiperparámetros de número de neuronas en la capa oculta, dropout, tasa de aprendizaje para el SGD, número de épocas, paciencia de entrenamiento, paciencia de la tasa de aprendizaje, factor, etc.

In [12]:
# Dictionary with parameters.
params = {
    # NeuralLanguageModel parameters.
    "vocab_size"     : max_vocab_size, # maximum vocabulary size.
    "model_order"    : model_order,    # model order.
    "embedding_size" : 100,            # 100-dimensional embeddings.
    "hidden_size"    : 200,            # 200 neurons in the hidden layer.
    "dropout"        : 0.1,            # dropout.
    # Trainer parameters.
    "lr"             : 2.3e-1,         # learning rate.
    "num_epochs"     : 100,            # number of epochs.
    "patience"       : 30,             # patience.
    "lr_patience"    : 10,             # learning rate patience.
    "lr_factor"      : 0.5,            # learning rate factor.
    "use_gpu"        : torch.cuda.is_available() # use GPU.
}

>Se entrenan los dos modelos (con y sin preinicializar con los embeddings).

In [14]:
# Model WITH embeddings.
model_with_embeddings = NeuralLanguageModel(
    params = params,                                         # parameters.
    pretrained_embeddings = ngram_data_emb.embeddings_matrix # pretrained embeddings.
)
if params["use_gpu"]:
    model_with_embeddings.cuda() # send model to GPU.

# Trainer for the model WITH embeddings.
trainer_with_embeddings = Trainer(
    model = model_with_embeddings,             # model.
    train_loader = train_loader_emb,           # train loader.
    val_loader = val_loader_emb,               # validation loader.
    params = params,                           # parameters.
    checkpoint_path = "model_with_embeddings", # checkpoint path.
    checkpoint_filename = "checkpoint.pth",    # checkpoint filename.
    best_model_filename = "best_model.pth"     # best model filename.
)
trainer_with_embeddings.train()

Training...:   1%|          | 1/100 [00:03<05:16,  3.20s/it]

 Epoch  1/100  ➤ Train acc: 0.1520 | Loss: 5.6509 | Val acc: 0.2109 | Epoch Time: 3.20 s | Total Time: 3.24 s


Training...:   2%|▏         | 2/100 [00:06<05:04,  3.11s/it]

 Epoch  2/100  ➤ Train acc: 0.1657 | Loss: 5.1622 | Val acc: 0.1242 | Epoch Time: 3.04 s | Total Time: 6.28 s


Training...:   3%|▎         | 3/100 [00:09<05:03,  3.13s/it]

...

 Epoch 29/100  ➤ Train acc: 0.3378 | Loss: 2.8554 | Val acc: 0.2030 | Epoch Time: 3.02 s | Total Time: 103.55 s


Training...:  30%|███       | 30/100 [01:46<04:17,  3.68s/it]

 Epoch 30/100  ➤ Train acc: 0.3365 | Loss: 2.8453 | Val acc: 0.1934 | Epoch Time: 3.06 s | Total Time: 106.61 s


Training...:  30%|███       | 30/100 [01:49<04:15,  3.65s/it]

No improvement. Breaking at epoch 30

--- Total Training Time: 106.61 seconds ---


In [15]:
# Model WITHOUT embeddings.
model_without_embeddings = NeuralLanguageModel(
    params = params,             # parameters.
    pretrained_embeddings = None # without pretrained embeddings.
)
if params["use_gpu"]:
    model_without_embeddings.cuda() # send model to GPU.

# Trainer for the model WITHOUT embeddings.
trainer_without_embeddings = Trainer(
    model = model_without_embeddings,             # model
    train_loader = train_loader_no_emb,           # train loader.
    val_loader = val_loader_no_emb,               # validation loader.
    params = params,                              # parameters.
    checkpoint_path = "model_without_embeddings", # checkpoint path.
    checkpoint_filename = "checkpoint.pth",       # checkpoint filename.
    best_model_filename = "best_model.pth"        # best model filename.
)
trainer_without_embeddings.train()

Training...:   1%|          | 1/100 [00:05<08:26,  5.11s/it]

 Epoch  1/100  ➤ Train acc: 0.1755 | Loss: 5.4800 | Val acc: 0.1729 | Epoch Time: 5.11 s | Total Time: 5.11 s


Training...:   2%|▏         | 2/100 [00:09<07:27,  4.57s/it]

 Epoch  2/100  ➤ Train acc: 0.1846 | Loss: 5.0483 | Val acc: 0.2293 | Epoch Time: 4.19 s | Total Time: 9.30 s


Training...:   3%|▎         | 3/100 [00:12<06:32,  4.05s/it]

...

 Epoch 30/100  ➤ Train acc: 0.3677 | Loss: 2.6875 | Val acc: 0.1890 | Epoch Time: 3.03 s | Total Time: 93.77 s


Training...:  31%|███       | 31/100 [01:36<03:24,  2.96s/it]

 Epoch 31/100  ➤ Train acc: 0.3706 | Loss: 2.6737 | Val acc: 0.1929 | Epoch Time: 2.93 s | Total Time: 96.71 s


Training...:  31%|███       | 31/100 [01:39<03:41,  3.21s/it]

No improvement. Breaking at epoch 31

--- Total Training Time: 96.71 seconds ---


> Una vez entrenados, se cargan los mejores modelos de ambas versiones:

In [16]:
# Load best model with embeddings.
best_model_with_embeddings = NeuralLanguageModel(
    params = params,                                         # parameters.
    pretrained_embeddings = ngram_data_emb.embeddings_matrix # pretrained embeddings.
)
best_model_with_embeddings.load_state_dict(torch.load('model_with_embeddings/best_model.pth', weights_only = True)['state_dict'])

# Load best model without embeddings.
best_model_without_embeddings = NeuralLanguageModel(
    params = params,             # parameters.
    pretrained_embeddings = None # without pretrained embeddings.
)
best_model_without_embeddings.load_state_dict(torch.load('model_without_embeddings/best_model.pth', weights_only = True)['state_dict'])

<All keys matched successfully>

>**Clase de evaluación:**
>
>Aquí se definen las funciones necesarias para evaluar a los modelos entrenados en el resto del ejercicio, se implementan los métodos:
>- *print_k_closest_words()*
>- *generate_sentence()* 
>- *log_likelihood()*
>- *evaluate_permutations()*
>- *calculate_perplexity()*
>
>(se describe cada uno en su docstring y un poco en su respectivo sub-ejercicio). Los primeros 3 métodos se implementan prácticamente igual a los vistos en la práctica anterior.

In [17]:
class Evaluation:
    @staticmethod
    def print_k_closest_words(embeddings: nn.Embedding, word: str, top_k: int, ngram_data: WordNgramData) -> None:
        """
        Print closest words method. It prints the top k closest words to the word provided.

        Parameters
        ----------
        embeddings : nn.Embedding
            Pre-trained embeddings from the model.
        word : str
            Word.
        top_k : int
            Number of closest words.
        ngram_data : WordNgramData
            N-gram data.

        Description
        -----------
        This method prints the k closest words to the word provided.
        """
        word_id = torch.LongTensor([ngram_data.w2id.get(word, ngram_data.w2id['<unk>'])]) # Get word ID and if not found, use <unk>.
        word_embedding = embeddings(word_id)                                              # Retrieves the embedding vector for the word from the model’s embedding layer.                              
        dists = torch.norm(embeddings.weight - word_embedding, dim = 1).detach()          # Calculate distances, embeddings.weight contains all word embeddings in the vocabulary.
        lst = sorted(enumerate(dists.numpy()), key = lambda x: x[1])                      # Sort by distance.
        for idx, difference in lst[1: top_k + 1]:                                         # Print the top k closest words.
            print(f"{ngram_data.id2w[idx]:<12} : {difference:.4f}")

    @staticmethod
    def parse_text(text: str, tokenizer: Callable, ngram_data: WordNgramData) -> Tuple[list, list]:
        """
        Parse text method. It recieves a text and returns the tokens and token ids.

        Parameters
        ----------
        text : str
            Text to parse.
        tokenizer : Callable
            Tokenizer.
        ngram_data : WordNgramData
            N-gram data.
        """
        all_tokens = [w.lower() if w in ngram_data.w2id else ngram_data.UNK for w in tokenizer(text)]     # Tokenizes the input text using tokenizer().
        token_ids = [ngram_data.w2id.get(w.lower(), ngram_data.w2id[ngram_data.UNK]) for w in all_tokens] # Maps tokens to their corresponding IDs.

        return all_tokens, token_ids                                                                      # Provides both the original tokens and their respective numeric IDs.

    @staticmethod
    def sample_next_word(raw_logits: np.ndarray, temperature: float = 0.1) -> int:
        """
        Sample next word method. It receives the raw logits and returns the next word.
        It returns the index of the next word in the vocabulary.

        Parameters
        ----------
        raw_logits : np.ndarray
            Raw logits (output of the neural network).
        temperature : float, optional
            Temperature, i.e., the higher the temperature, the more random the output.
            The less the temperature, the more deterministic the output. Default is 0.1.

        Returns
        -------
        int
            Next word (index in the vocabulary).
        """
        raw_logits = np.asarray(raw_logits).astype('float64') # convert raw logits to numpy array.
        preds = raw_logits / temperature                      # apply temperature scaling.
        exp_preds = np.exp(preds - np.max(preds))             # Numeric stability fix
        preds = exp_preds / np.sum(exp_preds)                 # nomalize the probabilities.

        return np.random.choice(len(preds), p = preds)        # randomly samples one word ID from the distribution.

    @staticmethod
    def predict_next_token(model: nn.Module, token_ids: list) -> int:
        """
        Predict next token method. It predicts the next token given the model and the token ids.

        Parameters
        ----------
        model : nn.Module
            Neural network model.
        token_ids : list
            Token ids list.

        Returns
        -------
        int
            Next token.
        """
        word_ids = torch.LongTensor(token_ids).unsqueeze(0)      # convert token IDs to tensor.
        raw_logits = model(word_ids).squeeze(0).detach().numpy() # raw predictions from the model.
        y_pred = Evaluation.sample_next_word(raw_logits, 1.0)    # to predict the next word.

        return y_pred

    @staticmethod
    def generate_sentence(model: nn.Module, initial_text: str, tokenizer: Callable, ngram_data: WordNgramData, max_length: int = 100) -> str:
        """
        This function generates a complete sentence using the trained model.
        It receives the initial text and predicts the next token word by word until the end of the sentence or a maximum length.

        Parameters
        ----------
        model : nn.Module
            Neural network model.
        initial_text : str
            Initial text to generate the sentence.
        tokenizer : Callable
            Tokenizer.
        ngram_data : WordNgramData
            N-gram data.
        max_length : int, optional
            Maximum length of the sentence. Default is 100.

        Returns
        -------
        str
            Generated sentence.
        """
        all_tokens, window_word_ids = Evaluation.parse_text(initial_text, tokenizer, ngram_data) # parse text.
        for _ in range(max_length):                                        # for each token.
            y_pred = Evaluation.predict_next_token(model, window_word_ids) # predict next token.
            next_word = ngram_data.id2w.get(y_pred, ngram_data.UNK)        # next word.
            all_tokens.append(next_word)                                   # append next word.

            # If the next word is the end of sentence, break.
            if next_word == ngram_data.EOS: 
                break
            window_word_ids.pop(0)         # remove first word.
            window_word_ids.append(y_pred) # append next word.
            
        return " ".join(all_tokens)        # return generated sentence.    
    
    @staticmethod
    def log_likelihood(model: nn.Module, text: str, ngram_model) -> float:
        """
        Log likelihood method. This function calculates how likely a given text is according to the model.

        Parameters
        ----------
        model : nn.Module
            Neural network model.
        text : str
            Text to calculate the log likelihood.
        ngram_model : WordNgramData
            N-gram model.

        Returns
        -------
        float
            Log likelihood. The higher the log likelihood, the more likely the text is according to the model.
        """
        X, y = ngram_model.transform([text]) # transform text.
        if len(X) < 2:                       # if less than 2 tokens.
            return -np.inf                   # return negative infinity.

        X, y = X[2:], y[2:]
        X = torch.LongTensor(X).unsqueeze(0) # tensor.

        logits = model(X).detach()                 # logits.
        probs = F.softmax(logits, dim = 1).numpy() # probabilities.

        return np.sum([np.log(probs[i][w]) for i, w in enumerate(y)]) # log likelihood.
    
    @staticmethod
    def evaluate_permutations(model: nn.Module, sentence: str, ngram_data, k: int = 5) -> None:
        """
        Evaluate permutations method. This function evaluates all possible permutations of a sentence
        and returns the k permutations with the highest log likelihood.

        Parameters
        ----------
        model : nn.Module
            Neural network model.
        sentence : str
            Sentence to evaluate.
        ngram_data : WordNgramData
            N-gram data.
        k : int
            Number of top permutations to return (default is 5).
        """
        tokens = sentence.split()  # split sentence.
        scores = []                # list to store permutations and scores.

        for perm in permutations(tokens):                    # for each permutation.
            perm_sentence = " ".join(perm)                   # join permutation.
            score = Evaluation.log_likelihood(model = model, # calculate log likelihood.
                                              text = perm_sentence,
                                              ngram_model = ngram_data)
            scores.append((perm_sentence, score))            # append permutation and score.

        top_k_permutations = sorted(scores, key = lambda x: x[1], reverse = True)[:k] # sort by score and get top k.

        print(f"Top {k} permutaciones con mayor log likelihood:")
        for i, (sentence, score) in enumerate(top_k_permutations, 1):
            print(f"{i}. {sentence} - Puntaje: {score:.2f}")

    @staticmethod
    def perplexity(model: nn.Module, data_loader: DataLoader) -> float:
        """
        Perplexity method. This function calculates the perplexity of the model using the validation data.
        
        Parameters
        ----------
        model : nn.Module
            Neural network model.
        data_loader : DataLoader
            Data loader with the validation data.
        
        Returns
        -------
        float
            Perplexity. The lower the perplexity, the better the model.
        """
        total_log_prob = 0 # total log probability.
        total_words = 0    # total words.
        model.eval()       # set model to evaluation mode.
    
        with torch.no_grad():
            for window_words, labels in data_loader:
                raw_logits = model(window_words)                                     # Get the raw logits from the model.
                log_probs = F.log_softmax(raw_logits, dim = 1)                       # Log softmax.
                labels = labels.unsqueeze(0) if labels.dim() == 0 else labels        # unsqueeze labels.
                total_log_prob += log_probs[range(len(labels)), labels].sum().item() # sum log probabilities.
                total_words += len(labels)                                           # total words.  
    
        return np.exp(-total_log_prob / total_words)

### **1.1.2**

> Con ayuda del método *print_k_closest_words()* de la clase *Evaluation*, se revisan las 10 palabras más similares, según su distancia, a las siguientes 3:

In [18]:
words_to_check = ["hola", "messi", "perro"]

# Model with embeddings.
print("-" * 50)
print("Modelo con embeddings preentrenados:")
print("-" * 50)
for word in words_to_check:
    print(f"\nPalabras similares a '{word}':")
    Evaluation.print_k_closest_words(embeddings = best_model_with_embeddings.embeddings,
                                     word = word.lower(),
                                     top_k = 10,
                                     ngram_data = ngram_data_emb)

# Model without embeddings.
print("")
print("-" * 50)
print("Modelo sin embeddings preentrenados:")
print("-" * 50)
for word in words_to_check:
    print(f"\nPalabras similares a '{word}':")
    Evaluation.print_k_closest_words(embeddings = best_model_without_embeddings.embeddings,
                                     word = word.lower(),
                                     top_k = 10,
                                     ngram_data = ngram_data_no_emb)

--------------------------------------------------
Modelo con embeddings preentrenados:
--------------------------------------------------

Palabras similares a 'hola':
hey          : 18.8399
oye          : 20.8158
pd           : 22.0726
saludame     : 23.0194
amig         : 23.0483
chaparro     : 23.1056
guapísima    : 23.1264
papasito     : 23.1564
gaby         : 23.3378
andrea       : 23.4096

Palabras similares a 'messi':
cristiano    : 10.7674
ronaldo      : 11.5831
benzema      : 12.8592
chicharito   : 13.6645
bale         : 14.7583
alemania     : 18.6632
holanda      : 19.2222
chicharo     : 19.2258
brasil       : 19.5027
portero      : 20.2453

Palabras similares a 'perro':
gato         : 9.5542
perrito      : 12.4943
enano        : 17.2279
gordo        : 17.4157
niño         : 17.5582
macho        : 17.8088
loro         : 18.2352
chamaco      : 18.2848
burro        : 18.3944
caballo      : 18.4915

--------------------------------------------------
Modelo sin embeddings preent

> Podríamos decir que, en general, el modelo con embeddings pre-entrenados da resultados más coherentes para todas las palabras en que fue probado.

### **1.1.3** Ponga al modelo a generar texto a partir de tres secuencias de inicio de su gusto.

> Con ayuda del método *generate_sentence()* de la clase *Evaluation*, se generaron oraciones de no más de 100 palabras, dadas las siguientes $3$ secuencias iniciales:
>
> Recordemos que estos métodos son prácticamente los mismos usados en la práctica.

In [19]:
initial_sentences = ["<s> <s> <s>", "yo opino que", "<s> hola como"]

# Model with embeddings.
print("-" * 50)
print("Modelo con embeddings preentrenados:")
print("-" * 50)
for start_text in initial_sentences:
    print(Evaluation.generate_sentence(model = best_model_with_embeddings,
                                       initial_text = start_text,
                                       tokenizer = tokenizer.tokenize,
                                       ngram_data = ngram_data_emb, 
                                       max_length = 100))

# Model without embeddings.
print("")
print("-" * 50)
print("Modelo sin embeddings preentrenados:")
print("-" * 50)
for start_text in initial_sentences:
    print(Evaluation.generate_sentence(model = best_model_without_embeddings,
                                       initial_text = start_text,
                                       tokenizer = tokenizer.tokenize,
                                       ngram_data = ngram_data_no_emb,
                                       max_length = 100))

--------------------------------------------------
Modelo con embeddings preentrenados:
--------------------------------------------------
<s> <s> <s> <unk> darnos con aparición de <unk> <unk> </s>
yo opino que me <unk> cabron <unk> <unk> 😒 <unk> <unk> <unk> por delgada y tuiter <unk> muy que <unk> con ): al mundial para <unk> y :d <unk> hdp <unk> <unk> <unk> </s>
<s> hola como <unk> <unk> putos <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> xd <unk> <unk> <unk> ojalá qué <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> yo no mamones <unk> </s>

--------------------------------------------------
Modelo sin embeddings preentrenados:
--------------------------------------------------
<s> <s> <s> economía mando a ese o diste <unk> <unk> <unk> <unk> <unk> estar me <unk> <unk> qué <unk> preparados favor <unk> 😌 <unk> mejor pedazos <unk> una países de en daniel <unk> te volver <unk> te pasas cagado <unk> de tuyo siempre 💅 <unk> </s>
yo opino que pasa digo corridos de dalas para q

> Desafortunadamente para ambos modelos (con y sin embeddings pre-entrenados), la presencia de palabras desconocidas afecta mucho pero, en general, las oraciones creadas sí tienen algo de sentido.

### **1.1.4** Escriba 5 ejemplos de oraciones y mídales el likelihood.

> Usando *log_likelihood()* como en la práctica pasada, se evaluaron las siguientes $5$ oraciones:

In [20]:
sentences = [
    "La ciencia de datos es aguacate",
    "La próxima vez que diga que",
    "Las redes neuronales son potentes",
    "analisis el de la de datos",
    "Nosotros si vamos al mundial"
]

# Model with embeddings.
print("-" * 50)
print("Likelihood con modelo con embeddings preentrenados:")
print("-" * 50)
for sentence in sentences:
    print(f"{sentence:<35} : {Evaluation.log_likelihood(model_with_embeddings, sentence, ngram_data_emb):.4f}")

# Model without embeddings.
print("")
print("-" * 50)
print("Likelihood con modelo sin embeddings preentrenados:")
print("-" * 50)
for sentence in sentences:
    print(f"{sentence:<35} : {Evaluation.log_likelihood(model_without_embeddings, sentence, ngram_data_no_emb):.4f}")

--------------------------------------------------
Likelihood con modelo con embeddings preentrenados:
--------------------------------------------------
La ciencia de datos es aguacate     : -29.8257
La próxima vez que diga que         : -16.4236
Las redes neuronales son potentes   : -19.7097
analisis el de la de datos          : -30.4697
Nosotros si vamos al mundial        : -9.7469

--------------------------------------------------
Likelihood con modelo sin embeddings preentrenados:
--------------------------------------------------
La ciencia de datos es aguacate     : -27.0425
La próxima vez que diga que         : -9.8327
Las redes neuronales son potentes   : -17.0673
analisis el de la de datos          : -33.6791
Nosotros si vamos al mundial        : -9.5223


> Intencionalmente coloqué oraciones sin sentido para averiguar que valor de likelihood se le asignaba y el resultado fue el esperado, valores muy malos para ambos modelos. Mientras que oraciones que incluso están en el corpus de entrenamiento, si fueron bien calificadas.

### **1.1.5** Proponga un ejemplo para ver estructuras sintácticas (permutaciones de palabras de alguna oración) buenas usando el likelihood a partir de una oración que usted proponga.

> Dada una oración propuesta, con ayuda del método *evaluate_permutations()*, se imprimieron las 5 permutaciones de esta oración con mejor puntaje usando el método *log_likelihood()*. La implementación de *evaluate_permutations()* es bastante intuible, primero se generan todas las permutaciones posibles, se calcula su likelihood, se ordenan y se imprimen las $k=5$ con mayor puntaje:

In [21]:
sentence_to_permute = "Nosotros si vamos al mundial <\s>"

print("-" * 50)
print("Modelo con embeddings preentrenados:")
print("-" * 50)
Evaluation.evaluate_permutations(model_with_embeddings, sentence_to_permute, ngram_data_emb)

print("")
print("-" * 50)
print("Modelo sin embeddings preentrenados:")
print("-" * 50)
Evaluation.evaluate_permutations(model_without_embeddings, sentence_to_permute, ngram_data_no_emb)

--------------------------------------------------
Modelo con embeddings preentrenados:
--------------------------------------------------
Top 5 permutaciones con mayor log likelihood:
1. Nosotros si vamos al mundial <\s> - Puntaje: -9.53
2. si Nosotros <\s> vamos al mundial - Puntaje: -13.93
3. Nosotros vamos al mundial <\s> si - Puntaje: -15.33
4. Nosotros si vamos <\s> al mundial - Puntaje: -15.55
5. si Nosotros vamos al mundial <\s> - Puntaje: -15.57

--------------------------------------------------
Modelo sin embeddings preentrenados:
--------------------------------------------------
Top 5 permutaciones con mayor log likelihood:
1. <\s> Nosotros si vamos al mundial - Puntaje: -9.86
2. si Nosotros <\s> vamos al mundial - Puntaje: -12.47
3. Nosotros si vamos al mundial <\s> - Puntaje: -13.06
4. mundial Nosotros si vamos al <\s> - Puntaje: -17.03
5. Nosotros si vamos <\s> al mundial - Puntaje: -19.63


> Los resultados son los esperados para un modelo decente, tiene mejor puntaje la permutación correcta y las que tienen muy poca variación.

### **1.1.6** Calcule la perplejidad del modelo sobre los datos val. Compárelo con la perplejidad del modelo de lenguaje sin embeddings preentrenados y el probabilista de la tarea anterior (el visto en clase). DISCUTA BREVEMENTE.

>A diferencia que en la tarea anterior, esta vez usamos a la perplejidad como:
>$$
>\text{perplexity} = \exp\left(-\frac{1}{N} \sum_{i=1}^N \log P(w_i | \text{contexto}) \right)
>$$
>Donde,
>- $N =$ Número total de palabras en el corpus.
>- $P(w_i | \text{contexto}) =$ Probabilidad predicha por el modelo para la palabra $w_i$ dado su contexto.
>- $\log P(w_i | \text{contexto}) =$ Log-probabilidad de la palabra correcta.
>
>Esto es estándar en modelos neuronales, ya que *torch.nn.functional.log_softmax()* calcula log-probabilidades en base e y en modelos de ngramas, se usa logaritmo base $2$.
>
> Con base en el método *calculate_perplexity()*, se calcula la perplejidad de los modelos sobre los datos de validación:

In [22]:
perplexity_with_embeddings = Evaluation.perplexity(model = best_model_with_embeddings, data_loader = val_loader_emb)
perplexity_without_embeddings = Evaluation.perplexity(model = best_model_without_embeddings, data_loader = val_loader_no_emb)

print(f"Perplejidad del modelo CON embeddings preentrenados: {perplexity_with_embeddings:.4f}")
print(f"Perplejidad del modelo SIN embeddings preentrenados: {perplexity_without_embeddings:.4f}")

Perplejidad del modelo CON embeddings preentrenados: 182.5393
Perplejidad del modelo SIN embeddings preentrenados: 119.9551


> El modelo sin embeddings preentrenados resultó mejor. En principio, se esperaría que el modelo con embeddings preentrenados tenga un mejor rendimiento (es decir, una menor perplejidad) porque parte de representaciones más ricas y entrenadas previamente. Yo pensaría que, si el vocabulario de los embeddings preentrenados no coincide bien con el vocabulario en el conjunto de entrenamiento, el modelo puede tener dificultades para encontrar representaciones significativas.


## **1.2- (50pts) Con base en la implementación mostrada en las prácticas del NLM,**
1. **Construya un modelo de lenguaje neuronal a nivel de carácter. Tomé en cuenta secuencias de tamaño 6 o más para el modelo, es decir hasta 5 caracteres o más en el contexto.**
2. **Ponga al modelo a generar texto 3 veces, con un máximo de 300 caracteres.**
3. **Escriba 5 ejemplos de oraciones y mídales el likelihood.**
4. **Escriba un ejemplo de estructura morfológica (permutaciones con caracteres) similar al de estructura sintáctica del profesor con 5 o más caracteres de su gusto (e.g., "ando ").**
5. **Calcule la perplejidad del modelo sobre los datos val. DISCUTA BREVEMENTE.**

> Comenzamos definiendo la clase *CharNgramData* adaptando la clase *WordNgramData* para procesar caracteres en lugar de palabras.
>
>Hay varias diferencias importantes:
>1. No considero importante limitar el tamaño del vocavulario a nivel de caracter porque es mucho más pequeño que el vocabulario de palabras.
>2. Como *default_tokenizer* puse uno que descompone el texto carácter por carácter y es el que uso en el ejercicio.
>3. Otras opciones como la de añadir embeddings y remover las stopwords también fueron eliminados porque ya no se manejan ese tipo de datos.
>4. Dentro de los métodos definidos, se hicieron ligeros cambios en
>       - *get_vocab()*
>       - *fit()*
>       - *get_ngram_doc()*
>       - *transform()*

In [23]:
class CharNgramData:
    """
    Character n-gram data class. It prepares the data for the character-level language model.
    It creates the vocabulary, word to index, and index to word mappings. It also transforms the data
    into n-grams.
    """
    def __init__(self, model_order: int = 6, tokenizer: Callable = None):
        """
        Constructor method for the CharNgramData class. If no tokenizer is provided, it uses the default tokenizer.

        Parameters
        ----------
        model_order : int, optional
            Model order. Default is 6.
        tokenizer : Callable, optional
            Tokenizer. Default is None
        """
        self.tokenizer = tokenizer or self.default_tokenizer  # Tokenizer.
        self.model_order = model_order                        # n-gram model order.
        self.UNK, self.SOS, self.EOS = "<unk>", "<s>", "</s>" # special tokens.
        self.vocab, self.w2id, self.id2w = None, None, None   # vocabulary, word to index, index to word mappings.

    @staticmethod
    def default_tokenizer(text: str) -> list:
        """
        Default tokenizer method. It tokenizes the text into characters.
        """
        return list(text)

    def get_vocab(self, corpus: list) -> set:
        """
        Get vocabulary method. It gets the vocabulary from the corpus, i.e., the set of unique characters
        and adds the special tokens.

        Parameters
        ----------
        corpus : list
            List of documents.
        """
        char_set = set("".join(corpus)) # set of unique characters.

        return char_set | {self.UNK, self.SOS, self.EOS}

    def fit(self, corpus: list) -> None:
        """
        Fit method. It fits the data, i.e., it builds the vocabulary and the word to index mappings.

        Parameters
        ----------
        corpus : list
            List of documents.
        """
        self.vocab = self.get_vocab(corpus)                            # get vocabulary.
        self.w2id = {char: idx for idx, char in enumerate(self.vocab)} # word to index.
        self.id2w = {idx: char for char, idx in self.w2id.items()}     # index to word.

    def get_ngram_doc(self, doc: str) -> list:
        """
        Get n-gram document method. It receives a document and returns the n-grams
        for the character-level language model.
        
        Parameters
        ----------
        doc : str
            Document.

        Returns
        -------
        list
            List of n-grams.
        """
        tokens = [char if char in self.vocab else self.UNK                 # if character is in the vocabulary.
                  for char in self.tokenizer(doc)]                         # tokenize the document.
        tokens = (self.model_order - 1) * [self.SOS] + tokens + [self.EOS] # add special tokens.

        return list(nltk.ngrams(tokens, self.model_order))                 # return n-grams.

    def transform(self, corpus: list) -> Tuple[np.ndarray, np.ndarray]:
        """
        Transform method. It transforms the data into n-grams for the character-level language model.

        Parameters
        ----------
        corpus : list
            List of documents.

        Returns
        -------
        np.ndarray
            X n-grams.
        np.ndarray
            y n-grams.
        """
        X_ngrams, y_ngrams = [], []                                   # n-grams.
        for doc in corpus:                                            # for each document.                 
            for chars_window in self.get_ngram_doc(doc):              # for each n-gram.
                char_ids = [self.w2id[char] for char in chars_window] # get character IDs.
                X_ngrams.append(char_ids[:-1])                        # X n-gram.
                y_ngrams.append(char_ids[-1])                         # y n-gram.
                
        return np.array(X_ngrams), np.array(y_ngrams)                 # return X and y n-grams.

### **1.2.1.**

> Para adaptar el modelo *NeuralLanguageModel* para recibir secuencias de caracteres y manejar adecuadamente la entrada, se define la clase *CharNeuralLanguageModel*

In [24]:
class CharNeuralLanguageModel(nn.Module):
    """
    Character-level neural language model class.
    """
    def __init__(self, params: dict):
        """
        Constructor method for the CharNeuralLanguageModel class.
        
        Parameters
        ----------
        params : dict
            Dictionary with parameters.

        Dictionary params
        -----------------
        vocab_size : int
            Vocabulary size.
        model_order : int
            Model order.
        embedding_size : int
            Vector size for the embeddings (each character is embedded into a dense vector).
        hidden_size : int
            Number of neurons in the hidden layer.
        dropout : float
            Dropout.
        """
        super().__init__()
        self.window_size = params['model_order'] - 1                              # window size (context).
        self.embedding_size = params['embedding_size']                            # embedding size.
        self.embeddings = nn.Embedding(params['vocab_size'], self.embedding_size) # embeddings.

        # Linear layers apply transformations to predict the next word.
        self.fc1 = nn.Linear(self.window_size * self.embedding_size, params['hidden_size'])
        # Dropout layer to prevent overfitting.
        self.drop1 = nn.Dropout(p = params['dropout'])
        # Output layer.
        self.fc2 = nn.Linear(params['hidden_size'], params['vocab_size'], bias = False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward method. Embeds input tokens into dense vectors. Reshapes them before feeding
        into the dense layers. Uses ReLU activation and Dropout before the final prediction.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
            
        Returns
        -------
        torch.Tensor
            Output tensor.
        """
        x = self.embeddings(x).view(-1, self.window_size * self.embedding_size) # Embeddings.
        
        return self.fc2(self.drop1(F.relu(self.fc1(x))))                        # Forward pass.

> Para el proceso de entrenamiento se usan los mismos métodos de las clases *UtilsForEvaluation* y *Trainer*, con sus hiperparámetros:
>- CrossEntropyLoss como métrica,
>- El optimizador SGD,
>- ReduceLROnPlateau como learning rate scheduler
>
>El orden del modelo es $6$ para tener hasta $5$ caracteres de contexto. Para la creación de los batchs, se mantiene un tamaño de $64$:

In [25]:
model_order = 6  # model order.
batch_size  = 64 # batch size.

> Dado esto, se crea la instancia del modelo de caracteres, estos datos pasan por los métodos *fit()* y *transform()* y se crean sus Data Loaders para datos de entrenamiento y validación:

In [27]:
# CharNgramData.
ngram_data_char = CharNgramData(
    model_order = model_order,                  # model order.
    tokenizer = CharNgramData.default_tokenizer # default tokenizer.
)

# Fit the CharNgramData and transform the data.
ngram_data_char.fit(X_train)                                    # fit the data.
X_train_char, y_train_char = ngram_data_char.transform(X_train) # transform train data.
X_val_char, y_val_char = ngram_data_char.transform(X_val)       # transform validation data.
print(f"Vocabulary size    : {len(ngram_data_char.vocab)}")      # vocabulary size.
print(f"X_train_char shape : {X_train_char.shape}")              # X train shape.
print(f"y_train_char shape : {y_train_char.shape}")              # y train shape.

# Dataset and DataLoader.
train_dataset_char = TensorDataset(torch.tensor(X_train_char), torch.tensor(y_train_char)) # train dataset.
train_loader_char = DataLoader(dataset = train_dataset_char, # training data loader.
                               batch_size = batch_size,      # batch size.
                               shuffle = True)               # shuffle.

val_dataset_char = TensorDataset(torch.tensor(X_val_char), torch.tensor(y_val_char)) # validation dataset.
val_loader_char = DataLoader(dataset = val_dataset_char, # validation data loader.
                             batch_size = batch_size,    # batch size.
                             shuffle = False)            # no shuffle.

Vocabulary size    : 428
X_train_char shape : (468758, 5)
y_train_char shape : (468758,)


>Se usan los mismos hiperparámetros de número de neuronas en la capa oculta, dropout, tasa de aprendizaje para el SGD, número de épocas, paciencia de entrenamiento, paciencia de la tasa de aprendizaje, factor, y sobre todo, el tamaño de embeddings.

In [28]:
# Dictionary with parameters.
params_char = {
    # CharNeuralLanguageModel parameters.
    "vocab_size"     : len(ngram_data_char.vocab), # vocabulary size.
    "model_order"    : model_order,                # model order.
    "embedding_size" : 100,                        # 100-dimensional embeddings.
    "hidden_size"    : 200,                        # 200 neurons in the hidden layer.
    "dropout"        : 0.1,                        # dropout.
    # Trainer parameters.
    "lr"             : 2.3e-1,                     # learning rate.
    "num_epochs"     : 100,                        # number of epochs.
    "patience"       : 20,                         # patience.
    "lr_patience"    : 10,                         # learning rate patience.
    "lr_factor"      : 0.5,                        # learning rate factor.
    "use_gpu"        : torch.cuda.is_available()   # use GPU.
}

>Se entrena el modelo.

In [29]:
# Model.
model_char = CharNeuralLanguageModel(
    params = params_char # parameters.
)
if params_char['use_gpu']:
    model_char.cuda() # send model to GPU.

# Trainer.
trainer_char = Trainer(
    model = model_char,                     # model.
    train_loader = train_loader_char,       # train loader.
    val_loader = val_loader_char,           # validation loader.
    params = params_char,                   # parameters.
    checkpoint_path = "model_char",         # checkpoint path.
    checkpoint_filename = "checkpoint.pth", # checkpoint filename.
    best_model_filename = "best_model.pth"  # best model filename.
)
trainer_char.train()

Training...:   1%|          | 1/100 [00:05<08:43,  5.28s/it]

 Epoch  1/100  ➤ Train acc: 0.4058 | Loss: 2.1332 | Val acc: 0.4420 | Epoch Time: 5.28 s | Total Time: 5.28 s


Training...:   2%|▏         | 2/100 [00:10<08:26,  5.17s/it]

 Epoch  2/100  ➤ Train acc: 0.4482 | Loss: 1.9393 | Val acc: 0.4561 | Epoch Time: 5.08 s | Total Time: 10.37 s


Training...:   3%|▎         | 3/100 [00:15<08:12,  5.08s/it]

...

 Epoch 98/100  ➤ Train acc: 0.5360 | Loss: 1.5671 | Val acc: 0.5179 | Epoch Time: 5.27 s | Total Time: 524.68 s


Training...:  99%|█████████▉| 99/100 [08:49<00:05,  5.39s/it]

 Epoch 99/100  ➤ Train acc: 0.5361 | Loss: 1.5678 | Val acc: 0.5180 | Epoch Time: 5.13 s | Total Time: 529.81 s


Training...: 100%|██████████| 100/100 [08:54<00:00,  5.35s/it]

 Epoch100/100  ➤ Train acc: 0.5365 | Loss: 1.5673 | Val acc: 0.5182 | Epoch Time: 5.10 s | Total Time: 534.91 s

--- Total Training Time: 534.91 seconds ---


> Una vez entrenado, se carga el mejor modelo:

In [30]:
# Load best model.
best_model_char = CharNeuralLanguageModel(
    params = params_char # parameters.
)
best_model_char.load_state_dict(torch.load('model_char/best_model.pth', weights_only = True)['state_dict'])

<All keys matched successfully>

>**Clase de evaluación:**
>
>Aquí se definen las funciones necesarias para evaluar al modelo entrenado en el resto del ejercicio, se implementan los métodos:
>- *generate_char_text()*
>- *log_likelihood()* 
>- *evaluate_permutations()*
>- *perplexity()*
>
>(se describe cada uno en su docstring y un poco en su respectivo sub-ejercicio).

In [31]:
class CharEvaluation:
    
    @staticmethod
    def generate_char_text(model: torch.nn.Module, initial_text: str, ngram_data: CharNgramData, max_length: int = 300) -> str:
        """
        Generates text up to max_length characters, ensuring the input matches the expected window size.

        Parameters
        ----------
        model : torch.nn.Module
            Model.
        initial_text : str
            Initial text.
        ngram_data : CharNgramData
            N-gram data.
        max_length : int, optional
            Maximum length. Default is 300.

        Returns
        -------
        str
            Generated text.
        """
        all_tokens = list(initial_text) # Initializate the list of tokens.
        window_word_ids = [ngram_data.w2id.get(char, ngram_data.w2id[ngram_data.UNK]) for char in initial_text]
    
        # Siz of the window is less than the model's window size.
        if len(window_word_ids) < model.window_size:
            window_word_ids = [ngram_data.w2id[ngram_data.SOS]] * (model.window_size - len(window_word_ids)) + window_word_ids
        elif len(window_word_ids) > model.window_size:
            window_word_ids = window_word_ids[-model.window_size:] # Take the last n characters.
    
        # Generate text.
        for _ in range(max_length):
            word_ids = torch.LongTensor(window_word_ids).unsqueeze(0)
            raw_logits = model(word_ids).squeeze(0).detach().numpy()
    
            next_char_id = np.random.choice(len(raw_logits), p = F.softmax(torch.tensor(raw_logits), dim = 0).numpy())
            next_char = ngram_data.id2w.get(next_char_id, ngram_data.UNK)
    
            all_tokens.append(next_char)
            if next_char == ngram_data.EOS:
                break
            
            window_word_ids.pop(0)  
            window_word_ids.append(next_char_id)
    
        return "".join(all_tokens)

    @staticmethod
    def log_likelihood(model: torch.nn.Module, text: str, ngram_data: CharNgramData) -> float:
        """
        Calculates the log-likelihood for a given sentence.

        Parameters
        ----------
        model : torch.nn.Module
            Model.
        text : str
            Text.
        ngram_data : CharNgramData
            N-gram data.

        Returns
        -------
        float
            Log-likelihood.
        """
        X, y = ngram_data.transform([text])
        if len(X) < 2:
            return -np.inf

        X, y = torch.LongTensor(X), torch.LongTensor(y)
        logits = model(X).detach()
        probs = F.softmax(logits, dim = 1).numpy()

        return np.sum([np.log(probs[i][w]) for i, w in enumerate(y)])

    @staticmethod
    def evaluate_permutations(model: torch.nn.Module, sequence: str, ngram_data: CharNgramData, k: int = 5) -> None:
        """
        Evaluates and prints the top-k permutations with the highest log-likelihood.

        Parameters
        ----------
        model : torch.nn.Module
            Model.
        sequence : str
            Sequence.
        ngram_data : CharNgramData
            N-gram data.
        k : int, optional
            Number of permutations. Default is 5.
        """
        chars = list(sequence.strip())  # Split characters
        scores = []

        for perm in permutations(chars):
            perm_sequence = "".join(perm)
            score = CharEvaluation.log_likelihood(model, perm_sequence, ngram_data)
            scores.append((perm_sequence, score))

        top_k_permutations = sorted(scores, key=lambda x: x[1], reverse=True)[:k]

        print(f"\nTop {k} permutaciones con mayor log-likelihood:")
        for i, (perm, score) in enumerate(top_k_permutations, 1):
            print(f"{i}. {perm} - Puntaje: {score:.4f}")

### **1.2.2**

In [32]:
print("-" * 50)
print("Generación de texto (Modelo de caracteres):")
print("-" * 50)
initial_sentences = ["<s> <s> <s>", "yo opino que", "<s> hola como"]

for start_text in initial_sentences:
    print(CharEvaluation.generate_char_text(
        model = best_model_char,
        initial_text = start_text,
        ngram_data = ngram_data_char,
        max_length = 300
    ))

--------------------------------------------------
Generación de texto (Modelo de caracteres):
--------------------------------------------------
<s> <s> <s></s>
yo opino que terduron 1.9</s>
<s> hola como muy fwimar 😣</s>


### **1.2.3**

In [33]:
sentences = [
    "La ciencia de datos es aguacate",
    "La próxima vez que diga que",
    "Las redes neuronales son potentes",
    "analisis el de la de datos",
    "Nosotros si vamos al mundial"
]

print("-" * 50)
print("Likelihood (Modelo de caracteres):")
print("-" * 50)
for sentence in sentences:
    print(f"{sentence:<35} : {CharEvaluation.log_likelihood(best_model_char, sentence, ngram_data_char):.4f}")

--------------------------------------------------
Likelihood (Modelo de caracteres):
--------------------------------------------------
La ciencia de datos es aguacate     : -71.2834
La próxima vez que diga que         : -39.6233
Las redes neuronales son potentes   : -89.6184
analisis el de la de datos          : -59.6748
Nosotros si vamos al mundial        : -51.2645


### **1.2.4**

In [34]:
morph_sequence = "ando "

print("-" * 50)
print("Permutaciones morfológicas (Modelo de caracteres):")
print("-" * 50)
CharEvaluation.evaluate_permutations(
    model = best_model_char,
    sequence = morph_sequence,
    ngram_data = ngram_data_char,
    k = 5
)

--------------------------------------------------
Permutaciones morfológicas (Modelo de caracteres):
--------------------------------------------------

Top 5 permutaciones con mayor log-likelihood:
1. ando - Puntaje: -14.0805
2. nado - Puntaje: -15.5232
3. noda - Puntaje: -18.4332
4. onda - Puntaje: -20.6968
5. dona - Puntaje: -21.3032


### **1.2.5**

In [35]:
perplexity_char = Evaluation.perplexity(
    model = best_model_char,
    data_loader = val_loader_char
)

print(f"Perplejidad del modelo de caracteres: {perplexity_char:.4f}")

Perplejidad del modelo de caracteres: 5.6714


> Se obtuvieron mucho mejores resultados destacando la perplejidad, que bajó de $243.4382$ y $123.1592$ a $5.7316$. Este resultado demuestra que el modelo a nivel de caracteres logra una representación más efectiva del lenguaje en este contexto, destacando su capacidad para manejar secuencias cortas y patrones morfológicos de manera más precisa.